# 110 — SmolVLA online per-chunk U20-Q75 gated plain refinement

Diagnostic counterpart to notebook 108 over the same 400 standard-LIBERO indices 0–9 identities. At every decision boundary it computes the same `(1,2,3)/(3,1,1)` pair-weighted stock U20 and uses the same frozen per-suite Q75 thresholds. A firing gate returns the completed plain P&P-refined chunk; a rejected gate returns the exact stock chunk. There is no averaging or consensus projection. The policy replans after 10 actions and videos are off.

In [ ]:
# Install NVIDIA EGL before importing MuJoCo.
import re, subprocess
driver = subprocess.check_output(
    ['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],
    text=True).splitlines()[0].strip()
package = f"libnvidia-gl-{re.match(r'\d+', driver).group()}"
if subprocess.run('ldconfig -p | grep -q libEGL_nvidia', shell=True).returncode:
    print('Installing NVIDIA EGL package:', package)
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',package], check=True)
subprocess.run('ldconfig -p | grep libEGL_nvidia', shell=True, check=True)

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pnp.smolvla_u20_gated_refinement_experiment import (
    SMOLVLA_U20_GATED_REFINEMENT_EXPERIMENT,
    SMOLVLA_U20_Q75_BY_SUITE,
    run_smolvla_u20_gated_refinement_eval_worker)

ROLLOUT_BATCH_SIZE = 1  # frozen for exact per-boundary stock fallback
print({'experiment': SMOLVLA_U20_GATED_REFINEMENT_EXPERIMENT,
       'cohort': 'standard LIBERO indices 0-9', 'identities': 400,
       'arm': 'per-chunk stock U20-Q75 -> plain refine, else exact stock',
       'thresholds': SMOLVLA_U20_Q75_BY_SUITE,
       'pnp_steps': [1, 2, 3], 'pnp_k_by_step': [3, 1, 1],
       'gate_horizon': 20, 'integration_steps': 10,
       'n_action_steps': 10, 'online_per_chunk': True,
       'print_every_identities': 10, 'video': 'off'})
run_smolvla_u20_gated_refinement_eval_worker(
    rollout_batch_size=ROLLOUT_BATCH_SIZE)